#     Aurora Oval / Polar View

In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Aurora Oval / Polar View
# Top-down Earth aurora visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "aurora_oval"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

EARTH_X = WIDTH * 0.5
EARTH_Y = HEIGHT * 0.52

EARTH_R = 140

AURORA_R_INNER = 78
AURORA_R_OUTER = 108

STAR_COUNT = 520
PARTICLE_COUNT = 850

ROTATION_SPEED = 0.22


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):

    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.5, 1.8),
        rng.uniform(20, 110),
        rng.uniform(0.4, 1.2),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:

        tw = (
            0.55
            + 0.45 * np.sin(
                phase * 2 * np.pi * speed + ph
            ) ** 2
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# EARTH
# ============================================================

def draw_earth(base, phase):

    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= EARTH_R

    shade = np.clip(
        1.0 - r / EARTH_R,
        0,
        1,
    )

    rotation = phase * 2 * np.pi * ROTATION_SPEED

    texture = (
        0.55
        + 0.18 * np.sin(dx * 0.04 + rotation)
        + 0.14 * np.sin(dy * 0.06 - rotation * 1.5)
        + 0.10 * np.sin((dx + dy) * 0.03)
    )

    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.42 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 20 + 25 * texture, 0)
    arr[..., 1] = np.where(sphere, 70 + 80 * texture, 0)
    arr[..., 2] = np.where(sphere, 130 + 90 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")

    layer.alpha_composite(img)

    # atmosphere
    d = ImageDraw.Draw(layer)

    for scale, alpha in [
        (1.12, 24),
        (1.24, 10),
    ]:

        rr = EARTH_R * scale

        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(120, 220, 255, alpha),
            width=3,
        )

    add_glow(base, layer, blur=8)


# ============================================================
# AURORA OVAL
# ============================================================

def draw_aurora(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)
    theta = np.arctan2(dy, dx)

    ring_center = (
        AURORA_R_INNER
        + AURORA_R_OUTER
    ) * 0.5

    ring_width = (
        AURORA_R_OUTER
        - AURORA_R_INNER
    ) * 0.5

    turbulence = (
        0.55
        + 0.25 * np.sin(theta * 7 + phase * 7)
        + 0.18 * np.sin(theta * 13 - phase * 11)
        + 0.12 * np.sin(theta * 19 + phase * 15)
    )

    turbulence = np.clip(turbulence, 0, 1)

    ring = np.exp(
        -((r - ring_center) ** 2)
        / (2 * ring_width ** 2)
    )

    alpha = ring * turbulence

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = 80
    arr[..., 1] = 255
    arr[..., 2] = 160
    arr[..., 3] = np.clip(alpha * 180, 0, 255).astype(np.uint8)

    img = Image.fromarray(arr, "RGBA")

    layer.alpha_composite(img)

    # brighter filaments
    for k in range(22):

        t = np.linspace(0, 2 * np.pi, 260)

        rr = (
            ring_center
            + rng.normal(0, 5)
        )

        rr += (
            8 * np.sin(
                t * rng.uniform(3, 7)
                + phase * rng.uniform(5, 10)
                + k
            )
        )

        x = EARTH_X + rr * np.cos(t)
        y = EARTH_Y + rr * np.sin(t)

        pts = list(zip(x, y))

        pulse = (
            0.5
            + 0.5 * np.sin(
                phase * 2 * np.pi * rng.uniform(2, 6)
                + k
            )
        )

        d.line(
            pts,
            fill=(120, 255, 190, int(50 + 120 * pulse)),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=16)


# ============================================================
# PARTICLE PRECIPITATION
# ============================================================

particles = []

for _ in range(PARTICLE_COUNT):

    particles.append((
        rng.uniform(0, 2 * np.pi),
        rng.uniform(0.0, 1.0),
        rng.uniform(0.4, 1.5),
        rng.choice([-1, 1]),
    ))


def draw_particles(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for angle, offset, speed, sign in particles:

        t = (
            offset
            + phase * speed
        ) % 1.0

        r0 = EARTH_R * 1.8
        r1 = AURORA_R_OUTER

        rr = r0 * (1 - t) + r1 * t

        theta = (
            angle
            + 0.18 * np.sin(
                phase * 2 * np.pi * 3 + angle
            )
        )

        x = EARTH_X + rr * np.cos(theta)
        y = EARTH_Y + sign * rr * np.sin(theta)

        alpha = int(
            110
            * (1 - abs(t - 0.65))
        )

        if alpha <= 0:
            continue

        size = 1.0 + 1.5 * (1 - t)

        d.ellipse(
            [
                x - size,
                y - size,
                x + size,
                y + size,
            ],
            fill=(180, 255, 220, alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# FIELD LINES
# ============================================================

def draw_field_lines(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k in range(16):

        angle = (
            k / 16
        ) * 2 * np.pi

        t = np.linspace(0, 1, 220)

        rr = (
            EARTH_R
            + 140 * t ** 1.4
        )

        bend = (
            0.18
            * np.sin(
                phase * 2 * np.pi * 2
                + angle * 3
            )
        )

        theta = angle + bend * t

        x = EARTH_X + rr * np.cos(theta)
        y = EARTH_Y + rr * np.sin(theta)

        pts = list(zip(x, y))

        d.line(
            pts,
            fill=(90, 220, 255, 36),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=3)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_field_lines(base, phase)

    draw_particles(base, phase)

    draw_earth(base, phase)

    draw_aurora(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Aurora Oval")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Aurora Oval
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/519065315.py:142: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/519065315.py:220: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/aurora_oval/aurora_oval.gif


In [5]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Aurora Oval / Mid-Latitude Orbital View
# Elliptical auroral oval, no particles
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "aurora_oval_midlatitude"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

EARTH_X = WIDTH * 0.5
EARTH_Y = HEIGHT * 0.62
EARTH_R = 205

# Aurora geometry: shifted upward and flattened by perspective
AURORA_X = EARTH_X
AURORA_Y = EARTH_Y - 105

AURORA_RX_INNER = 95
AURORA_RX_OUTER = 135

AURORA_RY_INNER = 48
AURORA_RY_OUTER = 82

AURORA_VERTICAL_HEIGHT = 30

STAR_COUNT = 520
FILAMENT_COUNT = 48
ROTATION_SPEED = 0.16


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.5, 1.7),
        rng.uniform(18, 95),
        rng.uniform(0.35, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):
    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:
        tw = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * speed + ph) ** 2

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# EARTH
# ============================================================

def draw_earth(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)
    sphere = r <= EARTH_R

    shade = np.clip(1.0 - r / EARTH_R, 0, 1)

    rotation = phase * 2 * np.pi * ROTATION_SPEED

    texture = (
        0.52
        + 0.18 * np.sin(dx * 0.030 + rotation)
        + 0.14 * np.sin(dy * 0.044 - rotation * 1.5)
        + 0.10 * np.sin((dx + dy) * 0.025)
    )
    texture = np.clip(texture, 0, 1)

    limb_darkening = shade ** 0.50
    intensity = limb_darkening * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    # Dark night-side Earth, not bright daylight globe
    arr[..., 0] = np.where(sphere, 8 + 26 * texture, 0)
    arr[..., 1] = np.where(sphere, 28 + 65 * texture, 0)
    arr[..., 2] = np.where(sphere, 70 + 110 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.alpha_composite(img)

    d = ImageDraw.Draw(layer)

    # Atmosphere / limb
    for scale, alpha, width in [
        (1.01, 70, 2),
        (1.05, 32, 3),
        (1.12, 12, 4),
    ]:
        rr = EARTH_R * scale
        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(95, 190, 255, alpha),
            width=width,
        )

    add_glow(base, layer, blur=8)


# ============================================================
# CITY LIGHTS / NIGHT-SIDE DETAIL
# ============================================================

city_lights = []

for _ in range(180):
    # Bias lights to lower visible hemisphere
    angle = rng.uniform(-2.8, -0.25)
    rr = EARTH_R * rng.uniform(0.15, 0.88)

    x = EARTH_X + rr * np.cos(angle)
    y = EARTH_Y - rr * np.sin(angle) * 0.65

    if (x - EARTH_X) ** 2 + (y - EARTH_Y) ** 2 <= EARTH_R ** 2:
        city_lights.append((
            x,
            y,
            rng.uniform(0.6, 1.8),
            rng.uniform(30, 120),
            rng.uniform(0, 2 * np.pi),
        ))


def draw_city_lights(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    for x, y, r, alpha, ph in city_lights:
        pulse = 0.75 + 0.25 * np.sin(phase * 2 * np.pi * 0.8 + ph) ** 2

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(255, 210, 120, int(alpha * pulse)),
        )

    add_glow(base, layer, blur=2)


# ============================================================
# AURORA OVAL
# ============================================================

filament_specs = []

for _ in range(FILAMENT_COUNT):
    filament_specs.append((
        rng.uniform(0, 2 * np.pi),
        rng.uniform(0.18, 0.55),
        rng.uniform(0.7, 1.25),
        rng.uniform(0.6, 1.4),
        rng.uniform(0, 2 * np.pi),
        rng.uniform(0.55, 1.0),
    ))


def ellipse_radius_field(dx, dy, rx, ry):
    return np.sqrt((dx / rx) ** 2 + (dy / ry) ** 2)


def draw_aurora_body(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - AURORA_X
    dy = yy - AURORA_Y

    inner_f = ellipse_radius_field(dx, dy, AURORA_RX_INNER, AURORA_RY_INNER)
    outer_f = ellipse_radius_field(dx, dy, AURORA_RX_OUTER, AURORA_RY_OUTER)

    # Approximate elliptical ring band
    ring_core = np.exp(-((outer_f - 1.0) ** 2) / (2 * 0.055 ** 2))
    ring_inner = np.exp(-((inner_f - 1.0) ** 2) / (2 * 0.070 ** 2))
    ring = np.maximum(ring_core, ring_inner * 0.72)

    theta = np.arctan2(dy / AURORA_RY_OUTER, dx / AURORA_RX_OUTER)

    turbulence = (
        0.52
        + 0.24 * np.sin(theta * 8 + phase * 8.0)
        + 0.18 * np.sin(theta * 15 - phase * 11.0)
        + 0.10 * np.sin(theta * 27 + phase * 16.0)
    )
    turbulence = np.clip(turbulence, 0, 1)

    # Keep aurora mostly above the visible limb / upper hemisphere
    visibility = smoothstep((EARTH_Y + 30 - yy) / 190)

    alpha = ring * turbulence * visibility

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)
    arr[..., 0] = 70
    arr[..., 1] = 255
    arr[..., 2] = 145
    arr[..., 3] = np.clip(alpha * 165, 0, 255).astype(np.uint8)

    img = Image.fromarray(arr, "RGBA")
    layer.alpha_composite(img)

    add_glow(base, layer, blur=18)


def draw_aurora_filaments(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k, (start, span, amp, speed, ph, brightness) in enumerate(filament_specs):
        t = np.linspace(start, start + span, 110)

        wobble = (
            1.0
            + 0.035 * np.sin(t * 9 + phase * 2 * np.pi * speed + ph)
            + 0.020 * np.sin(t * 21 - phase * 7 + k)
        )

        rx = ((AURORA_RX_INNER + AURORA_RX_OUTER) * 0.5) * wobble
        ry = ((AURORA_RY_INNER + AURORA_RY_OUTER) * 0.5) * wobble

        x = AURORA_X + rx * np.cos(t)
        y = AURORA_Y + ry * np.sin(t)

        pulse = 0.45 + 0.55 * np.sin(phase * 2 * np.pi * speed + ph) ** 2

        alpha = int((60 + 130 * pulse) * brightness)

        d.line(
            list(zip(x, y)),
            fill=(120, 255, 175, alpha),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=10)


def draw_aurora_curtains(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k, (start, span, amp, speed, ph, brightness) in enumerate(filament_specs):
        if k % 2:
            continue

        n = 7
        for j in range(n):
            a = start + span * j / max(1, n - 1)

            wobble = 1.0 + 0.035 * np.sin(a * 11 + phase * 8 + ph)

            x0 = AURORA_X + AURORA_RX_OUTER * wobble * np.cos(a)
            y0 = AURORA_Y + AURORA_RY_OUTER * wobble * np.sin(a)

            curtain_h = AURORA_VERTICAL_HEIGHT * (
                0.45
                + 0.55 * np.sin(phase * 2 * np.pi * speed + ph + j) ** 2
            )

            # Vertical sheets rise from the oval into space
            x1 = x0 + 8 * np.sin(phase * 4 + j + k)
            y1 = y0 - curtain_h

            alpha = int(35 + 85 * brightness)

            d.line(
                [(x0, y0), (x1, y1)],
                fill=(95, 255, 150, alpha),
                width=2,
            )

    add_glow(base, layer, blur=14)


# ============================================================
# OPTIONAL MAGNETIC GUIDE LINES
# ============================================================

def draw_field_guides(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k in range(14):
        a = 2 * np.pi * k / 14 + phase * 0.15

        t = np.linspace(0, 1, 160)

        rx0 = AURORA_RX_OUTER
        ry0 = AURORA_RY_OUTER

        x0 = AURORA_X + rx0 * np.cos(a)
        y0 = AURORA_Y + ry0 * np.sin(a)

        x = x0 + 55 * t * np.sin(a) * 0.25
        y = y0 - 130 * t

        alpha = int(18 + 18 * np.sin(phase * 2 * np.pi + k) ** 2)

        d.line(
            list(zip(x, y)),
            fill=(85, 215, 255, alpha),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=4)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_field_guides(base, phase)

    draw_earth(base, phase)
    draw_city_lights(base, phase)

    draw_aurora_body(base, phase)
    draw_aurora_filaments(base, phase)
    draw_aurora_curtains(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Aurora Oval Mid-Latitude View")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Aurora Oval Mid-Latitude View
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/232765107.py:141: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/232765107.py:264: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/aurora_oval_midlatitude/aurora_oval_midlatitude.gif


# Van Allen Belts / Radiation Belts

In [6]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Van Allen Belts / Radiation Belts
# Stylized Earth radiation belt visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "van_allen_belts"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

EARTH_X = WIDTH * 0.50
EARTH_Y = HEIGHT * 0.54
EARTH_R = 72

STAR_COUNT = 520

INNER_BELT_RX = 170
INNER_BELT_RY = 55

OUTER_BELT_RX = 290
OUTER_BELT_RY = 95

BELT_PARTICLES = 1800

IMPACT_PHASE = 0.52
ROTATION_SPEED = 0.18


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 1.6),
        rng.uniform(18, 105),
        rng.uniform(0.35, 1.15),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):
    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:
        tw = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * speed + ph) ** 2

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# EARTH
# ============================================================

def draw_earth(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)
    sphere = r <= EARTH_R

    shade = np.clip(1.0 - r / EARTH_R, 0, 1)
    rotation = phase * 2 * np.pi * ROTATION_SPEED

    texture = (
        0.52
        + 0.22 * np.sin(dx * 0.070 + rotation)
        + 0.16 * np.sin(dy * 0.060 - rotation * 1.3)
        + 0.10 * np.sin((dx + dy) * 0.040)
    )
    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.48 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 16 + 22 * texture, 0)
    arr[..., 1] = np.where(sphere, 56 + 95 * texture, 0)
    arr[..., 2] = np.where(sphere, 115 + 110 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.alpha_composite(img)

    d = ImageDraw.Draw(layer)

    for scale, alpha, width in [
        (1.10, 36, 2),
        (1.26, 14, 3),
    ]:
        rr = EARTH_R * scale
        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(100, 220, 255, alpha),
            width=width,
        )

    add_glow(base, layer, blur=8)


# ============================================================
# RADIATION BELT FIELDS
# ============================================================

def draw_belt_volume(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    storm = smoothstep((phase - IMPACT_PHASE) / 0.24)
    pulse = 0.5 + 0.5 * np.sin(phase * 2 * np.pi * 3.0) ** 2

    inner_scale = 1.0 + 0.10 * storm * pulse
    outer_scale = 1.0 + 0.18 * storm * pulse

    belts = [
        (
            INNER_BELT_RX * inner_scale,
            INNER_BELT_RY * inner_scale,
            (255, 200, 80),
            0.70,
        ),
        (
            OUTER_BELT_RX * outer_scale,
            OUTER_BELT_RY * outer_scale,
            (80, 235, 255),
            1.00,
        ),
    ]

    for rx, ry, col, alpha_mul in belts:
        for width, alpha in [
            (36, 12),
            (22, 22),
            (10, 45),
            (3, 95),
        ]:
            d.ellipse(
                [
                    EARTH_X - rx,
                    EARTH_Y - ry,
                    EARTH_X + rx,
                    EARTH_Y + ry,
                ],
                outline=(
                    col[0],
                    col[1],
                    col[2],
                    int(alpha * alpha_mul * (1.0 + 0.5 * storm)),
                ),
                width=width,
            )

    add_glow(base, layer, blur=14)


# ============================================================
# FIELD LINES
# ============================================================

def draw_field_lines(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    storm = smoothstep((phase - IMPACT_PHASE) / 0.22)

    for k in range(20):
        angle = 2 * np.pi * k / 20

        t = np.linspace(-1.0, 1.0, 260)

        rx = 105 + 210 * (abs(np.cos(angle)) ** 0.45)
        ry = 52 + 92 * (abs(np.sin(angle)) ** 0.35)

        rx *= 1.0 + 0.10 * storm
        ry *= 1.0 + 0.18 * storm

        theta = t * np.pi

        x = EARTH_X + rx * np.sin(theta) * np.cos(angle)
        y = EARTH_Y + ry * np.cos(theta)

        alpha = int(18 + 22 * np.sin(phase * 2 * np.pi * 2 + k) ** 2)

        d.line(
            list(zip(x, y)),
            fill=(90, 220, 255, alpha),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=4)


# ============================================================
# TRAPPED PARTICLES
# ============================================================

belt_particles = []

for _ in range(BELT_PARTICLES):
    belt_id = 0 if rng.random() < 0.45 else 1

    angle = rng.uniform(0, 2 * np.pi)
    offset = rng.uniform(0, 2 * np.pi)
    speed = rng.uniform(0.35, 1.75)

    radial_jitter = rng.normal(0.0, 0.045)
    brightness = rng.uniform(0.35, 1.0)
    size = rng.uniform(0.6, 1.7)

    belt_particles.append((
        belt_id,
        angle,
        offset,
        speed,
        radial_jitter,
        brightness,
        size,
    ))


def draw_trapped_particles(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    storm = smoothstep((phase - IMPACT_PHASE) / 0.22)
    injection = np.exp(-((phase - (IMPACT_PHASE + 0.08)) ** 2) / (2 * 0.045 ** 2))

    for belt_id, angle, offset, speed, radial_jitter, brightness, size in belt_particles:
        if belt_id == 0:
            rx = INNER_BELT_RX
            ry = INNER_BELT_RY
            color = (255, 220, 120)
            storm_gain = 0.35
        else:
            rx = OUTER_BELT_RX
            ry = OUTER_BELT_RY
            color = (150, 245, 255)
            storm_gain = 0.70

        scale = 1.0 + storm_gain * 0.20 * storm

        theta = angle + phase * 2 * np.pi * speed + offset

        jitter = 1.0 + radial_jitter + 0.025 * np.sin(phase * 9 + offset)

        x = EARTH_X + rx * scale * jitter * np.cos(theta)
        y = EARTH_Y + ry * scale * jitter * np.sin(theta)

        # hide particles behind Earth disk
        if (x - EARTH_X) ** 2 + (y - EARTH_Y) ** 2 < (EARTH_R * 0.96) ** 2:
            continue

        alpha = int(
            np.clip(
                (40 + 95 * brightness) * (1.0 + 1.6 * injection * storm_gain),
                0,
                255,
            )
        )

        rr = size * (1.0 + 0.6 * injection)

        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(color[0], color[1], color[2], alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# CME / STORM FRONT
# ============================================================

def draw_storm_front(base, phase):
    storm_t = smoothstep((phase - 0.34) / 0.22)

    if storm_t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    front_x = -120 + storm_t * (EARTH_X - OUTER_BELT_RX * 0.62)

    t = np.linspace(-1.25, 1.25, 360)

    x = front_x + 28 * np.cos(t) ** 2
    y = EARTH_Y + 260 * np.sin(t) * 0.62

    pts = list(zip(x, y))

    alpha_base = int(85 * (1.0 - abs(storm_t - 0.72)))

    if alpha_base > 0:
        for width, alpha in [
            (26, 10),
            (14, 24),
            (4, alpha_base),
        ]:
            d.line(
                pts,
                fill=(180, 255, 255, alpha),
                width=width,
                joint="curve",
            )

    add_glow(base, layer, blur=14)


# ============================================================
# BELT COMPRESSION / DISTURBANCE WAVE
# ============================================================

def draw_disturbance_wave(base, phase):
    storm = smoothstep((phase - IMPACT_PHASE) / 0.18)

    if storm <= 0.02:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    pulse = 1.0 - smoothstep((phase - IMPACT_PHASE) / 0.38)
    if pulse <= 0:
        return

    for scale, alpha in [
        (1.00, 75),
        (1.16, 34),
        (1.32, 16),
    ]:
        rx = OUTER_BELT_RX * scale * (1.0 + 0.12 * storm)
        ry = OUTER_BELT_RY * scale * (1.0 + 0.18 * storm)

        d.ellipse(
            [
                EARTH_X - rx,
                EARTH_Y - ry,
                EARTH_X + rx,
                EARTH_Y + ry,
            ],
            outline=(255, 245, 170, int(alpha * pulse)),
            width=3,
        )

    add_glow(base, layer, blur=12)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_storm_front(base, phase)

    draw_field_lines(base, phase)
    draw_belt_volume(base, phase)
    draw_trapped_particles(base, phase)
    draw_disturbance_wave(base, phase)

    draw_earth(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Van Allen Belts")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")
    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Van Allen Belts
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/1299919338.py:135: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/van_allen_belts/van_allen_belts.gif


This animation visualizes Earth’s Van Allen radiation belts — vast toroidal regions of trapped charged particles confined by the planet’s magnetic field. The inner and outer belts pulse and expand as a simulated geomagnetic disturbance arrives from space, illustrating how solar storms can inject energy into near-Earth space and destabilize radiation populations around the planet. Streams of energetic particles drift along magnetic field structures while the belts brighten and deform under the impact of the incoming CME-driven shock.

# Magnetotail Reconnection

In [8]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Magnetotail Snap / Magnetic Reconnection
# Simplified staged visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "magnetotail_snap"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

EARTH_X = 170
EARTH_Y = HEIGHT * 0.50
EARTH_R = 62

X_POINT_X = 560
X_POINT_Y = EARTH_Y

TAIL_END_X = 980

RECONNECTION_PHASE = 0.52

STAR_COUNT = 420
PLASMA_PARTICLES = 520


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


def gaussian_pulse(phase, center, width):
    return np.exp(-((phase - center) ** 2) / (2 * width ** 2))


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.35, 1.45),
        rng.uniform(16, 90),
        rng.uniform(0.35, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):
    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:
        tw = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * speed + ph) ** 2

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(175, 215, 255, int(a * tw)),
        )


# ============================================================
# EARTH
# ============================================================

def draw_earth(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)
    sphere = r <= EARTH_R

    shade = np.clip(1.0 - r / EARTH_R, 0, 1)

    texture = (
        0.52
        + 0.20 * np.sin(dx * 0.075 + phase * 2.0)
        + 0.16 * np.sin(dy * 0.060 - phase * 1.6)
        + 0.08 * np.sin((dx + dy) * 0.040)
    )
    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.52 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 10 + 24 * texture, 0)
    arr[..., 1] = np.where(sphere, 52 + 86 * texture, 0)
    arr[..., 2] = np.where(sphere, 110 + 116 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    img = Image.fromarray(arr.astype(np.uint8), "RGBA")
    layer.alpha_composite(img)

    d = ImageDraw.Draw(layer)

    for scale, alpha, width in [
        (1.08, 32, 2),
        (1.22, 12, 3),
    ]:
        rr = EARTH_R * scale
        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(90, 220, 255, alpha),
            width=width,
        )

    add_glow(base, layer, blur=7)


# ============================================================
# MAIN MAGNETOTAIL LINES
# ============================================================

def tail_line_points(side: int, offset: float, phase: float):
    """
    side: +1 upper line, -1 lower line
    offset: line distance from center
    """

    pull_t = smoothstep((phase - 0.18) / 0.30)
    recon_t = smoothstep((phase - RECONNECTION_PHASE) / 0.16)
    snap_t = smoothstep((phase - (RECONNECTION_PHASE + 0.07)) / 0.18)

    x = np.linspace(EARTH_X + EARTH_R * 0.8, TAIL_END_X, 360)

    base_y = EARTH_Y + side * offset

    # Before reconnection: lines are stretched and pulled into the X-point.
    pinch = np.exp(-((x - X_POINT_X) ** 2) / (2 * 95 ** 2))
    y = base_y - side * pinch * (54 * pull_t)

    # Tail waviness.
    y += side * 10 * np.sin((x - EARTH_X) * 0.018 + phase * 2 * np.pi * 1.2)

    # After reconnection, tail side beyond X-point snaps outward.
    right_mask = smoothstep((x - X_POINT_X) / 130)
    y += side * right_mask * snap_t * 42

    # Earthward branch bends inward after reconnection.
    left_mask = 1.0 - smoothstep((x - X_POINT_X + 40) / 180)
    y -= side * left_mask * snap_t * 22

    return list(zip(x, y))


def draw_tail_lines(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    flash = gaussian_pulse(phase, RECONNECTION_PHASE, 0.035)

    specs = [
        (+1, 78, (95, 230, 255), 115),
        (-1, 78, (95, 230, 255), 115),
        (+1, 118, (70, 185, 255), 72),
        (-1, 118, (70, 185, 255), 72),
    ]

    for side, offset, color, alpha in specs:
        pts = tail_line_points(side, offset, phase)

        d.line(
            pts,
            fill=(color[0], color[1], color[2], int(alpha + 55 * flash)),
            width=3 if offset < 100 else 2,
            joint="curve",
        )

    add_glow(base, layer, blur=8)


# ============================================================
# PLASMA SHEET
# ============================================================

def draw_plasma_sheet(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - X_POINT_X
    dy = yy - X_POINT_Y

    pull_t = smoothstep((phase - 0.18) / 0.30)
    recon_flash = gaussian_pulse(phase, RECONNECTION_PHASE, 0.060)

    width = 50 - 22 * pull_t + 20 * recon_flash

    sheet = np.exp(-(dy ** 2) / (2 * width ** 2))

    fade_left = smoothstep((xx - (EARTH_X + 80)) / 140)
    fade_right = 1.0 - smoothstep((xx - 910) / 80)

    turbulence = (
        0.52
        + 0.18 * np.sin(xx * 0.020 + phase * 8)
        + 0.14 * np.sin(xx * 0.035 - phase * 11)
    )

    alpha = sheet * fade_left * fade_right * turbulence

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)
    arr[..., 0] = 70
    arr[..., 1] = 180
    arr[..., 2] = 255
    arr[..., 3] = np.clip(alpha * (34 + 70 * recon_flash), 0, 255).astype(np.uint8)

    layer.alpha_composite(Image.fromarray(arr, "RGBA"))

    add_glow(base, layer, blur=18)


# ============================================================
# X-POINT / RECONNECTION FLASH
# ============================================================

def draw_x_point(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    flash = gaussian_pulse(phase, RECONNECTION_PHASE, 0.026)
    build = smoothstep((phase - 0.25) / 0.24)

    x = X_POINT_X
    y = X_POINT_Y

    # subtle X marker before event
    arm = 30 + 18 * build
    alpha = int(70 * build + 180 * flash)

    d.line(
        [(x - arm, y - arm), (x + arm, y + arm)],
        fill=(180, 255, 255, alpha),
        width=2,
    )
    d.line(
        [(x - arm, y + arm), (x + arm, y - arm)],
        fill=(180, 255, 255, alpha),
        width=2,
    )

    for rr, a in [
        (135, 12),
        (72, 36),
        (34, 120),
    ]:
        if flash > 0.02:
            d.ellipse(
                [x - rr, y - rr, x + rr, y + rr],
                fill=(130, 255, 255, int(a * flash * 4.0)),
            )

    add_glow(base, layer, blur=16)


# ============================================================
# PLASMOID EJECTION
# ============================================================

def draw_plasmoid(base, phase):
    t = smoothstep((phase - RECONNECTION_PHASE) / 0.32)

    if t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    x = X_POINT_X + 55 + 340 * t
    y = X_POINT_Y + 8 * np.sin(t * np.pi * 2)

    rx = 38 + 70 * t
    ry = 22 + 30 * t

    pulse = 0.6 + 0.4 * np.sin(phase * 2 * np.pi * 6) ** 2

    for scale, alpha, width in [
        (2.1, 14, 3),
        (1.45, 34, 3),
        (1.0, 95, 2),
    ]:
        d.ellipse(
            [
                x - rx * scale,
                y - ry * scale,
                x + rx * scale,
                y + ry * scale,
            ],
            outline=(120, 255, 255, int(alpha * pulse * (1.0 - 0.15 * t))),
            width=width,
        )

    add_glow(base, layer, blur=14)


# ============================================================
# EARTHWARD ENERGY BURST
# ============================================================

burst_particles = []

for _ in range(PLASMA_PARTICLES):
    burst_particles.append((
        rng.uniform(0.0, 1.0),
        rng.uniform(-1.0, 1.0),
        rng.uniform(0.8, 1.8),
        rng.uniform(0.7, 2.2),
    ))


def draw_earthward_burst(base, phase):
    t_global = smoothstep((phase - RECONNECTION_PHASE) / 0.28)

    if t_global <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for offset, spread, speed, size in burst_particles:
        t = (offset + t_global * speed) % 1.0

        x = X_POINT_X - 25 - 330 * t
        y = X_POINT_Y + spread * (18 + 24 * (1 - t))

        fade = 1.0 - abs(t - 0.48) * 2.0
        fade = np.clip(fade, 0, 1)

        alpha = int(120 * fade * (1.0 - 0.25 * t_global))

        if alpha <= 0:
            continue

        rr = size * (1.0 + 0.6 * fade)

        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(190, 255, 255, alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# SIMPLE LABELS / MARKERS
# ============================================================

def draw_stage_markers(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    build = smoothstep((phase - 0.20) / 0.30)
    flash = gaussian_pulse(phase, RECONNECTION_PHASE, 0.030)
    snap = smoothstep((phase - (RECONNECTION_PHASE + 0.08)) / 0.24)

    # Minimal HUD-like event markers, no text.
    x = X_POINT_X
    y = X_POINT_Y

    marker_alpha = int(40 + 90 * build + 80 * flash)

    d.rectangle(
        [x - 52, y - 52, x + 52, y + 52],
        outline=(120, 245, 255, marker_alpha),
        width=1,
    )

    if snap > 0:
        px = X_POINT_X + 55 + 340 * snap
        d.line(
            [(X_POINT_X + 20, y), (px - 45, y)],
            fill=(120, 245, 255, int(45 * (1 - snap))),
            width=1,
        )

    add_glow(base, layer, blur=5)


# ============================================================
# AURORA RESPONSE ON EARTH
# ============================================================

def draw_aurora_response(base, phase):
    t = smoothstep((phase - (RECONNECTION_PHASE + 0.10)) / 0.18)

    if t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    pulse = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * 5) ** 2

    for sign in [-1, 1]:
        ay = EARTH_Y + sign * EARTH_R * 0.54

        d.ellipse(
            [
                EARTH_X - EARTH_R * 0.72,
                ay - EARTH_R * 0.13,
                EARTH_X + EARTH_R * 0.72,
                ay + EARTH_R * 0.13,
            ],
            outline=(100, 255, 155, int(125 * t * pulse)),
            width=3,
        )

    add_glow(base, layer, blur=9)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_plasma_sheet(base, phase)
    draw_tail_lines(base, phase)
    draw_x_point(base, phase)

    draw_plasmoid(base, phase)
    draw_earthward_burst(base, phase)
    draw_stage_markers(base, phase)

    draw_earth(base, phase)
    draw_aurora_response(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Magnetotail Snap")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")
    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )
else:
    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Magnetotail Snap
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/147173600.py:257: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  layer.alpha_composite(Image.fromarray(arr, "RGBA"))
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/147173600.py:135: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/magnetotail_snap/magnetotail_snap.gif


This animation illustrates magnetic reconnection inside Earth’s magnetotail — the stretched nightside extension of the planet’s magnetic field. As solar-wind pressure builds, opposing magnetic field lines are forced together until they suddenly reconnect at the bright X-point deep in the tail. The released magnetic energy ejects a plasmoid outward into space while a burst of energized plasma races back toward Earth, triggering geomagnetic substorms and enhanced auroral activity near the poles.

# Parker Spiral / Interplanetary Magnetic Field

In [10]:
from __future__ import annotations







from pathlib import Path



import numpy as np



import imageio.v2 as imageio



from PIL import Image, ImageDraw, ImageFilter











# ============================================================



# Parker Spiral / Interplanetary Magnetic Field



# Solar wind + rotating IMF visualization



# ============================================================







OUTPUT_FORMAT = "webm"  # gif | mp4



FPS = 24



DURATION = 8



TOTAL_FRAMES = FPS * DURATION







WIDTH = 960



HEIGHT = 540







ANIMATION_NAME = "parker_spiral"







OUT_DIR = Path("media-site/animations") / ANIMATION_NAME



OUT_DIR.mkdir(parents=True, exist_ok=True)







OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"







BG = (0, 0, 0, 255)







RNG_SEED = 42



rng = np.random.default_rng(RNG_SEED)







SUN_X = 170



SUN_Y = HEIGHT * 0.52



SUN_R = 72







STAR_COUNT = 480







SPIRAL_COUNT = 8



FIELD_PARTICLES = 2200







CME_PHASE = 0.55







MAX_RADIUS = 1180











# ============================================================



# HELPERS



# ============================================================







def rgba():



    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))











def add_glow(base, layer, blur=8):



    glow = layer.filter(ImageFilter.GaussianBlur(blur))



    base.alpha_composite(glow)



    base.alpha_composite(layer)











def smoothstep(t):



    t = np.clip(t, 0.0, 1.0)



    return t * t * (3.0 - 2.0 * t)











# ============================================================



# BACKGROUND



# ============================================================







stars = []







for _ in range(STAR_COUNT):



    stars.append((



        rng.uniform(0, WIDTH),



        rng.uniform(0, HEIGHT),



        rng.uniform(0.4, 1.5),



        rng.uniform(18, 95),



        rng.uniform(0.35, 1.1),



        rng.uniform(0, 2 * np.pi),



    ))











def draw_background(base, phase):



    d = ImageDraw.Draw(base)







    for x, y, r, a, speed, ph in stars:



        tw = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * speed + ph) ** 2







        d.ellipse(



            [x-r, y-r, x+r, y+r],



            fill=(180, 220, 255, int(a * tw)),



        )











# ============================================================



# SUN



# ============================================================







def draw_sun(base, phase):



    layer = rgba()







    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]







    dx = xx - SUN_X



    dy = yy - SUN_Y







    r = np.sqrt(dx * dx + dy * dy)







    sphere = r <= SUN_R



    shade = np.clip(1.0 - r / SUN_R, 0, 1)







    texture = (



        0.55



        + 0.22 * np.sin(dx * 0.08 + phase * 8)



        + 0.18 * np.sin(dy * 0.06 - phase * 10)



        + 0.10 * np.sin((dx + dy) * 0.05)



    )







    texture = np.clip(texture, 0, 1)







    intensity = shade ** 0.42 * texture







    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)







    arr[..., 0] = np.where(sphere, 255, 0)



    arr[..., 1] = np.where(sphere, 145 + 90 * intensity, 0)



    arr[..., 2] = np.where(sphere, 40 + 60 * intensity, 0)



    arr[..., 3] = np.where(sphere, 255, 0)







    layer.alpha_composite(



        Image.fromarray(arr.astype(np.uint8), "RGBA")



    )







    d = ImageDraw.Draw(layer)







    for rr, alpha in [



        (145, 12),



        (112, 26),



        (88, 55),



    ]:



        d.ellipse(



            [



                SUN_X - rr,



                SUN_Y - rr,



                SUN_X + rr,



                SUN_Y + rr,



            ],



            fill=(255, 180, 70, alpha),



        )







    add_glow(base, layer, blur=18)











# ============================================================



# PARKER SPIRALS



# ============================================================







spiral_offsets = np.linspace(



    0,



    2 * np.pi,



    SPIRAL_COUNT,



    endpoint=False,



)











def parker_spiral(theta, offset, phase):



    """



    Simplified Parker spiral:



    radius grows with theta while solar rotation twists field lines.



    """







    radial_speed = 54



    rotation_rate = 0.82







    r = radial_speed * theta







    angle = (



        theta



        + offset



        + phase * rotation_rate * 2 * np.pi



    )







    x = SUN_X + r * np.cos(angle)



    y = SUN_Y + r * np.sin(angle)







    return x, y











def draw_spirals(base, phase):



    layer = rgba()



    d = ImageDraw.Draw(layer)







    for i, offset in enumerate(spiral_offsets):







        theta = np.linspace(0, 18, 1800)







        x, y = parker_spiral(theta, offset, phase)







        pts = []







        for xx, yy in zip(x, y):







            if -200 <= xx <= WIDTH + 200 and -200 <= yy <= HEIGHT + 200:



                pts.append((xx, yy))







        pulse = (



            0.6



            + 0.4 * np.sin(



                phase * 2 * np.pi * 2



                + i



            ) ** 2



        )







        alpha = int(55 + 70 * pulse)







        d.line(



            pts,



            fill=(90, 230, 255, alpha),



            width=2,



            joint="curve",



        )







    add_glow(base, layer, blur=8)











# ============================================================



# SOLAR WIND PARTICLES



# ============================================================







particles = []







for _ in range(FIELD_PARTICLES):







    particles.append((



        rng.integers(0, SPIRAL_COUNT),



        rng.uniform(0.0, 18.0),



        rng.uniform(0.2, 1.3),



        rng.uniform(0.5, 2.0),



        rng.uniform(0.4, 1.0),



    ))











def draw_particles(base, phase):



    layer = rgba()



    d = ImageDraw.Draw(layer)







    for spiral_id, theta0, speed, size, brightness in particles:







        theta = (



            theta0



            + phase * speed * 5.5



        ) % 18.0







        x, y = parker_spiral(



            theta,



            spiral_offsets[spiral_id],



            phase,



        )







        if not (-30 <= x <= WIDTH + 30 and -30 <= y <= HEIGHT + 30):



            continue







        alpha = int(



            40 + 120 * brightness



        )







        rr = size







        d.ellipse(



            [



                x - rr,



                y - rr,



                x + rr,



                y + rr,



            ],



            fill=(180, 255, 255, alpha),



        )







    add_glow(base, layer, blur=4)











# ============================================================



# CME SHOCK FRONT



# ============================================================







def draw_cme(base, phase):







    t = smoothstep(



        (phase - CME_PHASE) / 0.22



    )







    if t <= 0:



        return







    layer = rgba()



    d = ImageDraw.Draw(layer)







    rr = 120 + t * 820







    theta = np.linspace(



        -1.05,



        1.05,



        520,



    )







    x = SUN_X + rr * np.cos(theta)



    y = SUN_Y + rr * np.sin(theta)







    pts = list(zip(x, y))







    pulse = (



        0.7



        + 0.3 * np.sin(



            phase * 2 * np.pi * 5



        ) ** 2



    )







    for width, alpha in [



        (32, 10),



        (18, 22),



        (7, int(70 * pulse)),



    ]:



        d.line(



            pts,



            fill=(150, 255, 255, alpha),



            width=width,



            joint="curve",



        )







    add_glow(base, layer, blur=16)











# ============================================================



# PLANET ORBITS



# ============================================================







planet_specs = [



    (330, 18, (255, 210, 120)),



    (520, 24, (120, 180, 255)),



    (760, 30, (255, 140, 100)),



]











def draw_planets(base, phase):



    layer = rgba()



    d = ImageDraw.Draw(layer)







    for orbit_r, radius, color in planet_specs:







        angle = (



            phase * 2 * np.pi * (



                0.08



                + orbit_r / 9000



            )



        )







        x = SUN_X + orbit_r * np.cos(angle)



        y = SUN_Y + orbit_r * np.sin(angle)







        if not (-100 <= x <= WIDTH + 100 and -100 <= y <= HEIGHT + 100):



            continue







        d.ellipse(



            [



                x - radius,



                y - radius,



                x + radius,



                y + radius,



            ],



            fill=(



                color[0],



                color[1],



                color[2],



                255,



            ),



        )







        d.ellipse(



            [



                x - radius * 1.4,



                y - radius * 1.4,



                x + radius * 1.4,



                y + radius * 1.4,



            ],



            outline=(



                color[0],



                color[1],



                color[2],



                55,



            ),



            width=2,



        )







    add_glow(base, layer, blur=5)











# ============================================================



# IMF FLOW STREAMS



# ============================================================







def draw_flow_streams(base, phase):



    layer = rgba()



    d = ImageDraw.Draw(layer)







    for i, offset in enumerate(spiral_offsets):







        for k in range(12):







            theta0 = (



                phase * 5



                + k * 1.2



            ) % 18







            theta1 = theta0 + 0.35







            x0, y0 = parker_spiral(



                theta0,



                offset,



                phase,



            )







            x1, y1 = parker_spiral(



                theta1,



                offset,



                phase,



            )







            if (



                -50 <= x0 <= WIDTH + 50



                and -50 <= y0 <= HEIGHT + 50



            ):



                d.line(



                    [(x0, y0), (x1, y1)],



                    fill=(120, 240, 255, 80),



                    width=2,



                )







    # ------------------------------------------------------------



    # Opaque solar photosphere cap



    # ------------------------------------------------------------







    d.ellipse(



        [



            SUN_X - SUN_R,



            SUN_Y - SUN_R,



            SUN_X + SUN_R,



            SUN_Y + SUN_R,



        ],



        fill=(255, 170, 55, 255),



    )







    add_glow(base, layer, blur=4)











# ============================================================



# FRAME



# ============================================================







def render_frame(i):







    phase = i / TOTAL_FRAMES







    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)







    draw_background(base, phase)







    draw_spirals(base, phase)







    draw_flow_streams(base, phase)







    draw_particles(base, phase)







    draw_cme(base, phase)







    draw_planets(base, phase)







    draw_sun(base, phase)







    return np.array(base.convert("RGB"))











# ============================================================



# RENDER



# ============================================================







frames = []







print("[START] Parker Spiral")







for i in range(TOTAL_FRAMES):







    if i % FPS == 0:



        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")







    frames.append(render_frame(i))







if OUTPUT_FORMAT == "gif":



    imageio.mimsave(



        OUT_FILE,



        frames,



        fps=FPS,



        loop=0,



    )



elif OUTPUT_FORMAT == "webm":



    from vizlib.animation_export import export_animation



    export_animation(frames, OUT_DIR, ANIMATION_NAME, "webm", FPS)



    OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.webm"



else:



    imageio.mimsave(



        OUT_FILE,



        frames,



        fps=FPS,



        quality=9,



        macro_block_size=1,



    )







print()



print(f"[SAVED] {OUT_FILE.resolve()}")



[START] Parker Spiral
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/95852377.py:132: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/parker_spiral/parker_spiral.gif


This animation visualizes the Parker spiral — the large-scale structure of the interplanetary magnetic field carried outward by the solar wind. As the Sun rotates, magnetic field lines rooted in the solar corona are continuously dragged into space and twisted into enormous spiral arms that fill the Solar System. Streams of charged particles flow along these curved magnetic structures while a propagating CME shock disturbs the surrounding heliosphere, illustrating how solar activity can transport energy and plasma across interplanetary space.

#     Solar Wind Streamlines / Heliospheric Current Sheet

In [11]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# CME Propagation to Earth
# Solar eruption crossing interplanetary space
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "cme_propagation_to_earth"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

SUN_X = 150
SUN_Y = HEIGHT * 0.52
SUN_R = 74

EARTH_X = 790
EARTH_Y = HEIGHT * 0.52
EARTH_R = 42

STAR_COUNT = 480
PARTICLE_COUNT = 1500

CME_START_PHASE = 0.18
IMPACT_PHASE = 0.76


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 1.6),
        rng.uniform(18, 95),
        rng.uniform(0.35, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):
    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:

        tw = (
            0.55
            + 0.45 * np.sin(
                phase * 2 * np.pi * speed + ph
            ) ** 2
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# SUN
# ============================================================

def draw_sun(base, phase):

    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - SUN_X
    dy = yy - SUN_Y

    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= SUN_R
    shade = np.clip(1.0 - r / SUN_R, 0, 1)

    texture = (
        0.55
        + 0.22 * np.sin(dx * 0.08 + phase * 8)
        + 0.18 * np.sin(dy * 0.06 - phase * 10)
        + 0.10 * np.sin((dx + dy) * 0.05)
    )

    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.42 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 255, 0)
    arr[..., 1] = np.where(sphere, 145 + 90 * intensity, 0)
    arr[..., 2] = np.where(sphere, 40 + 60 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    layer.alpha_composite(
        Image.fromarray(arr.astype(np.uint8), "RGBA")
    )

    d = ImageDraw.Draw(layer)

    for rr, alpha in [
        (155, 10),
        (120, 22),
        (92, 50),
    ]:
        d.ellipse(
            [
                SUN_X - rr,
                SUN_Y - rr,
                SUN_X + rr,
                SUN_Y + rr,
            ],
            fill=(255, 180, 70, alpha),
        )

    # opaque photosphere
    d.ellipse(
        [
            SUN_X - SUN_R,
            SUN_Y - SUN_R,
            SUN_X + SUN_R,
            SUN_Y + SUN_R,
        ],
        fill=(255, 170, 55, 255),
    )

    add_glow(base, layer, blur=18)


# ============================================================
# EARTH
# ============================================================

def draw_earth(base, phase):

    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= EARTH_R

    shade = np.clip(
        1.0 - r / EARTH_R,
        0,
        1,
    )

    texture = (
        0.52
        + 0.18 * np.sin(dx * 0.12 + phase * 3)
        + 0.14 * np.sin(dy * 0.09 - phase * 4)
    )

    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.5 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 10 + 24 * texture, 0)
    arr[..., 1] = np.where(sphere, 55 + 90 * texture, 0)
    arr[..., 2] = np.where(sphere, 120 + 110 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    layer.alpha_composite(
        Image.fromarray(arr.astype(np.uint8), "RGBA")
    )

    d = ImageDraw.Draw(layer)

    # atmosphere
    for scale, alpha in [
        (1.12, 26),
        (1.28, 10),
    ]:

        rr = EARTH_R * scale

        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(90, 220, 255, alpha),
            width=3,
        )

    add_glow(base, layer, blur=6)


# ============================================================
# SOLAR WIND STREAMS
# ============================================================

def draw_solar_wind(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for y0 in np.linspace(
        HEIGHT * 0.18,
        HEIGHT * 0.82,
        18,
    ):

        t = np.linspace(0, 1, 220)

        x = SUN_X + t * (EARTH_X - SUN_X + 160)

        y = (
            y0
            + 12 * np.sin(
                t * 6
                + phase * 2 * np.pi * 1.5
                + y0 * 0.02
            )
        )

        pts = list(zip(x, y))

        d.line(
            pts,
            fill=(90, 210, 255, 22),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=4)


# ============================================================
# CME FRONT
# ============================================================

def draw_cme(base, phase):

    t = smoothstep(
        (phase - CME_START_PHASE)
        / (IMPACT_PHASE - CME_START_PHASE)
    )

    if t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = SUN_X + t * (EARTH_X - SUN_X)
    cy = SUN_Y

    # expanding shell
    rx = 90 + t * 240
    ry = 70 + t * 170

    pulse = (
        0.7
        + 0.3 * np.sin(
            phase * 2 * np.pi * 5
        ) ** 2
    )

    # front arc
    theta = np.linspace(
        -1.15,
        1.15,
        520,
    )

    x = cx + rx * np.cos(theta)
    y = cy + ry * np.sin(theta)

    pts = list(zip(x, y))

    for width, alpha in [
        (34, 10),
        (18, 22),
        (8, int(75 * pulse)),
    ]:

        d.line(
            pts,
            fill=(160, 255, 255, alpha),
            width=width,
            joint="curve",
        )

    # turbulent plasma body
    for k in range(28):

        angle = rng.uniform(
            -1.0,
            1.0,
        )

        rr = rng.uniform(
            rx * 0.2,
            rx * 0.95,
        )

        px = cx + rr * np.cos(angle)
        py = cy + rr * np.sin(angle) * 0.7

        size = rng.uniform(4, 12)

        alpha = int(
            rng.uniform(18, 65)
            * (1.0 - t * 0.2)
        )

        d.ellipse(
            [
                px - size,
                py - size,
                px + size,
                py + size,
            ],
            fill=(120, 240, 255, alpha),
        )

    add_glow(base, layer, blur=18)


# ============================================================
# ENERGETIC PARTICLES
# ============================================================

particles = []

for _ in range(PARTICLE_COUNT):

    particles.append((
        rng.uniform(0.0, 1.0),
        rng.uniform(-1.0, 1.0),
        rng.uniform(0.5, 1.6),
        rng.uniform(0.5, 2.0),
    ))


def draw_particles(base, phase):

    t = smoothstep(
        (phase - CME_START_PHASE)
        / (IMPACT_PHASE - CME_START_PHASE)
    )

    if t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for offset, spread, speed, size in particles:

        travel = (
            offset
            + t * speed
        ) % 1.0

        x = (
            SUN_X
            + travel * (EARTH_X - SUN_X)
        )

        y = (
            SUN_Y
            + spread * (
                30
                + 90 * t
            )
            + 12 * np.sin(
                phase * 2 * np.pi * 4
                + offset * 8
            )
        )

        alpha = int(
            110
            * (1 - abs(travel - 0.5))
        )

        if alpha <= 0:
            continue

        rr = size * (
            1.0
            + 0.5 * t
        )

        d.ellipse(
            [
                x - rr,
                y - rr,
                x + rr,
                y + rr,
            ],
            fill=(180, 255, 255, alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# EARTH IMPACT
# ============================================================

def draw_impact(base, phase):

    impact_t = smoothstep(
        (phase - IMPACT_PHASE)
        / 0.10
    )

    if impact_t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    pulse = (
        0.6
        + 0.4 * np.sin(
            phase * 2 * np.pi * 7
        ) ** 2
    )

    # bow shock flash
    for rr, alpha in [
        (130, 12),
        (82, 28),
        (48, 90),
    ]:

        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(
                140,
                255,
                255,
                int(alpha * impact_t * pulse),
            ),
            width=4,
        )

    # auroral response
    for sign in [-1, 1]:

        ay = (
            EARTH_Y
            + sign * EARTH_R * 0.62
        )

        d.ellipse(
            [
                EARTH_X - EARTH_R * 0.85,
                ay - EARTH_R * 0.16,
                EARTH_X + EARTH_R * 0.85,
                ay + EARTH_R * 0.16,
            ],
            outline=(
                120,
                255,
                170,
                int(140 * impact_t * pulse),
            ),
            width=3,
        )

    add_glow(base, layer, blur=12)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new(
        "RGBA",
        (WIDTH, HEIGHT),
        BG,
    )

    draw_background(base, phase)

    draw_solar_wind(base, phase)

    draw_cme(base, phase)

    draw_particles(base, phase)

    draw_impact(base, phase)

    draw_earth(base, phase)

    draw_sun(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] CME Propagation")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(
            f"[RENDER] frame {i}/{TOTAL_FRAMES}"
        )

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] CME Propagation
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/1879875127.py:215: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr.astype(np.uint8), "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/1879875127.py:140: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/cme_propagation_to_earth/cme_propagation_to_earth.gif


This animation shows a coronal mass ejection traveling from the Sun toward Earth through interplanetary space. The expanding plasma front carries magnetized material outward with the solar wind, crossing the heliosphere as energetic particles stream ahead of and within the shock structure. When the CME reaches Earth, the impact compresses the planet’s magnetic environment and triggers a bright geomagnetic response, including enhanced auroral activity near the polar regions.

# Heliospheric Current Sheet / Ballerina Skirt

In [12]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter


# ============================================================
# Heliospheric Current Sheet
# "Ballerina Skirt" IMF visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "heliospheric_current_sheet"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

SUN_X = 180
SUN_Y = HEIGHT * 0.50
SUN_R = 76

STAR_COUNT = 420

FIELD_PARTICLES = 2400
SECTOR_LINES = 9

CURRENT_SHEET_AMPLITUDE = 120
CURRENT_SHEET_WAVELENGTH = 240

CME_PHASE = 0.56


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):

    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.4, 1.5),
        rng.uniform(18, 90),
        rng.uniform(0.35, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:

        tw = (
            0.55
            + 0.45 * np.sin(
                phase * 2 * np.pi * speed + ph
            ) ** 2
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# SUN
# ============================================================

def draw_sun(base, phase):

    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - SUN_X
    dy = yy - SUN_Y

    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= SUN_R
    shade = np.clip(1.0 - r / SUN_R, 0, 1)

    texture = (
        0.55
        + 0.22 * np.sin(dx * 0.08 + phase * 8)
        + 0.18 * np.sin(dy * 0.06 - phase * 10)
        + 0.10 * np.sin((dx + dy) * 0.05)
    )

    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.42 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 255, 0)
    arr[..., 1] = np.where(sphere, 145 + 90 * intensity, 0)
    arr[..., 2] = np.where(sphere, 40 + 60 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    layer.alpha_composite(
        Image.fromarray(arr.astype(np.uint8), "RGBA")
    )

    d = ImageDraw.Draw(layer)

    for rr, alpha in [
        (155, 10),
        (120, 22),
        (92, 50),
    ]:
        d.ellipse(
            [
                SUN_X - rr,
                SUN_Y - rr,
                SUN_X + rr,
                SUN_Y + rr,
            ],
            fill=(255, 180, 70, alpha),
        )

    # opaque photosphere
    d.ellipse(
        [
            SUN_X - SUN_R,
            SUN_Y - SUN_R,
            SUN_X + SUN_R,
            SUN_Y + SUN_R,
        ],
        fill=(255, 170, 55, 255),
    )

    add_glow(base, layer, blur=18)


# ============================================================
# CURRENT SHEET
# ============================================================

def current_sheet_y(x, phase):

    radial = (
        x - SUN_X
    )

    wave = (
        np.sin(
            radial / CURRENT_SHEET_WAVELENGTH
            - phase * 2 * np.pi * 0.85
        )
    )

    secondary = (
        0.35
        * np.sin(
            radial / 120
            + phase * 2 * np.pi * 1.6
        )
    )

    return (
        SUN_Y
        + CURRENT_SHEET_AMPLITUDE
        * (wave + secondary)
    )


def draw_current_sheet(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    x = np.linspace(
        SUN_X + 10,
        WIDTH + 120,
        1600,
    )

    y = current_sheet_y(x, phase)

    pts = list(zip(x, y))

    # volumetric ribbon
    for width, alpha in [
        (54, 6),
        (34, 12),
        (18, 24),
        (7, 55),
    ]:

        d.line(
            pts,
            fill=(120, 240, 255, alpha),
            width=width,
            joint="curve",
        )

    # sector boundaries
    for k in range(SECTOR_LINES):

        offset = (
            k * 18
            - SECTOR_LINES * 9
        )

        y2 = y + offset

        pts2 = list(zip(x, y2))

        d.line(
            pts2,
            fill=(70, 180, 255, 20),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=14)


# ============================================================
# IMF FLOW PARTICLES
# ============================================================

particles = []

for _ in range(FIELD_PARTICLES):

    particles.append((
        rng.uniform(0.0, 1.0),
        rng.uniform(-1.0, 1.0),
        rng.uniform(0.4, 1.5),
        rng.uniform(0.6, 2.0),
    ))


def draw_particles(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for offset, spread, speed, size in particles:

        t = (
            offset
            + phase * speed
        ) % 1.0

        x = (
            SUN_X
            + t * (WIDTH - SUN_X + 160)
        )

        y0 = current_sheet_y(x, phase)

        y = (
            y0
            + spread * (
                12
                + 22 * np.sin(
                    phase * 2 * np.pi * 2
                    + offset * 7
                ) ** 2
            )
        )

        if not (
            -40 <= x <= WIDTH + 40
            and -40 <= y <= HEIGHT + 40
        ):
            continue

        alpha = int(
            35
            + 120 * (
                1
                - abs(spread)
            )
        )

        rr = size

        d.ellipse(
            [
                x - rr,
                y - rr,
                x + rr,
                y + rr,
            ],
            fill=(180, 255, 255, alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# CME DISTURBANCE
# ============================================================

def draw_cme(base, phase):

    t = smoothstep(
        (phase - CME_PHASE) / 0.22
    )

    if t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = SUN_X + t * 700

    theta = np.linspace(
        -1.2,
        1.2,
        600,
    )

    rx = 90 + t * 260
    ry = 65 + t * 180

    x = cx + rx * np.cos(theta)
    y = SUN_Y + ry * np.sin(theta)

    pts = list(zip(x, y))

    pulse = (
        0.7
        + 0.3 * np.sin(
            phase * 2 * np.pi * 6
        ) ** 2
    )

    for width, alpha in [
        (30, 10),
        (16, 24),
        (7, int(80 * pulse)),
    ]:

        d.line(
            pts,
            fill=(150, 255, 255, alpha),
            width=width,
            joint="curve",
        )

    add_glow(base, layer, blur=16)


# ============================================================
# HELIOSPHERIC SECTORS
# ============================================================

def draw_sector_regions(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    x = np.linspace(
        SUN_X,
        WIDTH,
        800,
    )

    y = current_sheet_y(x, phase)

    upper = list(zip(x, y - 60))
    lower = list(zip(x[::-1], (y + 60)[::-1]))

    poly = upper + lower

    d.polygon(
        poly,
        fill=(40, 120, 255, 14),
    )

    upper2 = list(zip(x, y - 140))
    lower2 = list(zip(x[::-1], (y - 60)[::-1]))

    poly2 = upper2 + lower2

    d.polygon(
        poly2,
        fill=(255, 120, 60, 10),
    )

    add_glow(base, layer, blur=18)


# ============================================================
# SOLAR WIND STREAMS
# ============================================================

def draw_streams(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k in range(22):

        t = np.linspace(0, 1, 280)

        x = (
            SUN_X
            + t * (WIDTH - SUN_X + 120)
        )

        base_y = current_sheet_y(x, phase)

        spread = (
            k - 11
        ) * 16

        y = (
            base_y
            + spread
            + 8 * np.sin(
                t * 7
                + phase * 2 * np.pi * 1.8
                + k
            )
        )

        pts = list(zip(x, y))

        d.line(
            pts,
            fill=(100, 220, 255, 16),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=4)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new(
        "RGBA",
        (WIDTH, HEIGHT),
        BG,
    )

    draw_background(base, phase)

    draw_sector_regions(base, phase)

    draw_streams(base, phase)

    draw_current_sheet(base, phase)

    draw_particles(base, phase)

    draw_cme(base, phase)

    draw_sun(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Heliospheric Current Sheet")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(
            f"[RENDER] frame {i}/{TOTAL_FRAMES}"
        )

    frames.append(render_frame(i))

if OUTPUT_FORMAT == "gif":

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        loop=0,
    )

else:

    imageio.mimsave(
        OUT_FILE,
        frames,
        fps=FPS,
        quality=9,
        macro_block_size=1,
    )

print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Heliospheric Current Sheet
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/3771438745.py:142: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr.astype(np.uint8), "RGBA")


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/heliospheric_current_sheet/heliospheric_current_sheet.gif


In [15]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter
from vizlib.animation_export import export_animation


# ============================================================
# Heliospheric Current Sheet — Side View
# Clear staged "ballerina skirt" visualization
# ============================================================

OUTPUT_FORMAT = "gif"  # gif | mp4
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "heliospheric_current_sheet_side_view"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

SUN_X = 145
SUN_Y = HEIGHT * 0.50
SUN_R = 72

EARTH_X = 820
EARTH_Y = HEIGHT * 0.50
EARTH_R = 34

STAR_COUNT = 380
WIND_PARTICLES = 900

SHEET_AMPLITUDE = 86
SHEET_WAVELENGTH = 260
SHEET_PHASE_SPEED = 0.85

CME_PHASE = 0.54


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


def sheet_y(x, phase):
    radial = x - SUN_X

    main = np.sin(
        radial / SHEET_WAVELENGTH
        - phase * 2 * np.pi * SHEET_PHASE_SPEED
    )

    fine = 0.28 * np.sin(
        radial / 95
        + phase * 2 * np.pi * 1.35
    )

    return SUN_Y + SHEET_AMPLITUDE * (main + fine)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):
    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.35, 1.45),
        rng.uniform(15, 90),
        rng.uniform(0.35, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):
    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:
        tw = 0.55 + 0.45 * np.sin(phase * 2 * np.pi * speed + ph) ** 2

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# SECTOR REGIONS
# ============================================================

def draw_sector_regions(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    x = np.linspace(SUN_X, WIDTH, 900)
    y = sheet_y(x, phase)

    upper_poly = (
        [(SUN_X, 0), (WIDTH, 0)]
        + list(zip(x[::-1], y[::-1]))
    )

    lower_poly = (
        list(zip(x, y))
        + [(WIDTH, HEIGHT), (SUN_X, HEIGHT)]
    )

    # Opposite IMF polarity sectors
    d.polygon(upper_poly, fill=(40, 110, 255, 24))
    d.polygon(lower_poly, fill=(255, 110, 60, 20))

    add_glow(base, layer, blur=18)


# ============================================================
# CURRENT SHEET RIBBON
# ============================================================

def draw_current_sheet(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    x = np.linspace(SUN_X + 15, WIDTH + 80, 1300)
    y = sheet_y(x, phase)

    pts = list(zip(x, y))

    # Thick, readable "ballerina skirt" edge-on sheet
    for width, alpha in [
        (70, 8),
        (46, 16),
        (24, 35),
        (8, 90),
    ]:
        d.line(
            pts,
            fill=(130, 245, 255, alpha),
            width=width,
            joint="curve",
        )

    # bright crest highlights
    for offset, alpha in [(-28, 24), (28, 24)]:
        pts2 = list(zip(x, y + offset))
        d.line(
            pts2,
            fill=(80, 180, 255, alpha),
            width=2,
            joint="curve",
        )

    add_glow(base, layer, blur=16)


# ============================================================
# SOLAR WIND STREAMLINES
# ============================================================

def draw_solar_wind_streams(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k in range(16):
        t = np.linspace(0, 1, 240)

        x = SUN_X + t * (WIDTH - SUN_X + 120)

        base_y = SUN_Y + (k - 7.5) * 28

        y = (
            base_y
            + 10 * np.sin(t * 7 + phase * 2 * np.pi * 1.8 + k)
            + 0.18 * (sheet_y(x, phase) - SUN_Y)
        )

        d.line(
            list(zip(x, y)),
            fill=(110, 220, 255, 22),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=4)


# ============================================================
# SOLAR WIND PARTICLES
# ============================================================

wind_particles = []

for _ in range(WIND_PARTICLES):
    wind_particles.append((
        rng.uniform(0.0, 1.0),
        rng.uniform(-1.0, 1.0),
        rng.uniform(0.4, 1.35),
        rng.uniform(0.5, 1.7),
    ))


def draw_solar_wind_particles(base, phase):
    layer = rgba()
    d = ImageDraw.Draw(layer)

    for offset, spread, speed, size in wind_particles:
        t = (offset + phase * speed) % 1.0

        x = SUN_X + t * (WIDTH - SUN_X + 120)

        y_center = sheet_y(x, phase)

        y = (
            y_center
            + spread * 72
            + 8 * np.sin(phase * 2 * np.pi * 2.5 + offset * 10)
        )

        if not (-40 <= x <= WIDTH + 40 and -40 <= y <= HEIGHT + 40):
            continue

        near_sheet = np.exp(-((y - y_center) ** 2) / (2 * 60 ** 2))

        alpha = int(35 + 115 * near_sheet)

        rr = size

        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(180, 255, 255, alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# SUN
# ============================================================

def draw_sun(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - SUN_X
    dy = yy - SUN_Y

    r = np.sqrt(dx * dx + dy * dy)
    sphere = r <= SUN_R
    shade = np.clip(1.0 - r / SUN_R, 0, 1)

    texture = (
        0.55
        + 0.22 * np.sin(dx * 0.08 + phase * 8)
        + 0.18 * np.sin(dy * 0.06 - phase * 10)
        + 0.10 * np.sin((dx + dy) * 0.05)
    )
    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.42 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 255, 0)
    arr[..., 1] = np.where(sphere, 145 + 90 * intensity, 0)
    arr[..., 2] = np.where(sphere, 40 + 60 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    layer.alpha_composite(Image.fromarray(arr.astype(np.uint8), "RGBA"))

    d = ImageDraw.Draw(layer)

    for rr, alpha in [
        (155, 10),
        (120, 22),
        (92, 50),
    ]:
        d.ellipse(
            [
                SUN_X - rr,
                SUN_Y - rr,
                SUN_X + rr,
                SUN_Y + rr,
            ],
            fill=(255, 180, 70, alpha),
        )

    # hard opaque photosphere cap
    d.ellipse(
        [
            SUN_X - SUN_R,
            SUN_Y - SUN_R,
            SUN_X + SUN_R,
            SUN_Y + SUN_R,
        ],
        fill=(255, 170, 55, 255),
    )

    add_glow(base, layer, blur=18)


# ============================================================
# EARTH
# ============================================================

def draw_earth(base, phase):
    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - EARTH_X
    dy = yy - EARTH_Y

    r = np.sqrt(dx * dx + dy * dy)
    sphere = r <= EARTH_R

    shade = np.clip(1.0 - r / EARTH_R, 0, 1)

    texture = (
        0.52
        + 0.18 * np.sin(dx * 0.12 + phase * 3)
        + 0.14 * np.sin(dy * 0.09 - phase * 4)
    )
    texture = np.clip(texture, 0, 1)

    intensity = shade ** 0.50 * texture

    arr = np.zeros((HEIGHT, WIDTH, 4), dtype=np.uint8)

    arr[..., 0] = np.where(sphere, 10 + 24 * texture, 0)
    arr[..., 1] = np.where(sphere, 55 + 90 * texture, 0)
    arr[..., 2] = np.where(sphere, 120 + 110 * intensity, 0)
    arr[..., 3] = np.where(sphere, 255, 0)

    layer.alpha_composite(Image.fromarray(arr.astype(np.uint8), "RGBA"))

    d = ImageDraw.Draw(layer)

    for scale, alpha in [(1.12, 26), (1.28, 10)]:
        rr = EARTH_R * scale
        d.ellipse(
            [
                EARTH_X - rr,
                EARTH_Y - rr,
                EARTH_X + rr,
                EARTH_Y + rr,
            ],
            outline=(90, 220, 255, alpha),
            width=3,
        )

    add_glow(base, layer, blur=6)


# ============================================================
# CME FRONT CROSSING THE SHEET
# ============================================================

def draw_cme_front(base, phase):
    t = smoothstep((phase - CME_PHASE) / 0.24)

    if t <= 0:
        return

    layer = rgba()
    d = ImageDraw.Draw(layer)

    cx = SUN_X + t * (EARTH_X - SUN_X + 80)
    cy = sheet_y(cx, phase)

    rx = 72 + t * 190
    ry = 60 + t * 120

    theta = np.linspace(-1.15, 1.15, 460)

    x = cx + rx * np.cos(theta)
    y = cy + ry * np.sin(theta)

    pulse = 0.7 + 0.3 * np.sin(phase * 2 * np.pi * 6) ** 2

    for width, alpha in [
        (34, 10),
        (18, 24),
        (7, int(78 * pulse)),
    ]:
        d.line(
            list(zip(x, y)),
            fill=(160, 255, 255, alpha),
            width=width,
            joint="curve",
        )

    add_glow(base, layer, blur=16)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):
    phase = i / TOTAL_FRAMES

    base = Image.new("RGBA", (WIDTH, HEIGHT), BG)

    draw_background(base, phase)

    draw_sector_regions(base, phase)
    draw_solar_wind_streams(base, phase)

    draw_current_sheet(base, phase)
    draw_solar_wind_particles(base, phase)

    draw_cme_front(base, phase)

    draw_earth(base, phase)
    draw_sun(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Heliospheric Current Sheet — Side View")

for i in range(TOTAL_FRAMES):
    if i % FPS == 0:
        print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")
    frames.append(render_frame(i))


print()
print(f"[SAVED] {OUT_FILE.resolve()}")

[START] Heliospheric Current Sheet — Side View
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/2948877407.py:364: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  layer.alpha_composite(Image.fromarray(arr.astype(np.uint8), "RGBA"))
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_2861/2948877407.py:298: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  layer.alpha_composite(Image.fromarray(arr.astype(np.uint8), "RGBA"))


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192

[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/heliospheric_current_sheet_side_view/heliospheric_current_sheet_side_view.gif
